# Member 2 notebook: E. coli, Hospital A 

**What this gives you:** the shared data loader and `run_cv`, the common test every method must use, so all results can be compared.

**Setup (once):** put `spectra_ecoli_A.npy` and `labels_ecoli_A.csv` in a folder named `shared_data` next to this notebook.

**Rules**
- Do not change `run_cv`, the folds or `MODEL_ABX`.
- Save each method as a CSV with exactly these columns: `antibiotic, method, auc, auc_std, prauc, n_features`.
- Give each method its own `method` name.

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.feature_selection import f_classif

# Load shared data from the shared_data folder
X = np.load("shared_data/spectra_ecoli_A.npy")
lab = pd.read_csv("shared_data/labels_ecoli_A.csv")

# Assigned antibiotics for Member 2
MODEL_ABX = ["Ciprofloxacin", "Piperacillin-Tazobactam", "Norfloxacin", "Ceftazidime"]

print("Data loaded successfully!")
print("Spectra shape:", X.shape, "| Labels shape:", lab.shape)

Data loaded successfully!
Spectra shape: (7320, 6000) | Labels shape: (7320, 32)


## `run_cv`: the common evaluation function

For one antibiotic: use the saved folds (train on 4, test on 1, repeat 5 times).
The reducer is fitted on the **training part only** (no leakage), then Logistic Regression is trained.
It returns AUC (Area Under the ROC Curve), PR-AUC (Precision-Recall AUC) and the number of features used.

In [2]:
def run_cv(reducer, antibiotic):
    """reducer = object with fit(X, y) / transform(X), or None for no reduction."""
    has_label = (lab[antibiotic].notna()) & (lab[f"fold_{antibiotic}"] >= 0)
    Xa = X[has_label.values]                                   # spectra with an R/S result
    ya = lab.loc[has_label, antibiotic].astype(int).values     # R=1, S=0
    fa = lab.loc[has_label, f"fold_{antibiotic}"].values       # fold number 0-4

    aucs, praucs, n_feats = [], [], []
    for k in range(5):
        tr, te = fa != k, fa == k                  # train = other 4 folds, test = fold k
        steps = [StandardScaler()]                 # put all bins on the same scale
        if reducer is not None:
            steps.append(clone(reducer))           # fresh copy, fitted on train only
        steps.append(LogisticRegression(max_iter=1000))   # simple classifier
        model = make_pipeline(*steps).fit(Xa[tr], ya[tr])

        p = model.predict_proba(Xa[te])[:, 1]      # probability of resistant
        aucs.append(roc_auc_score(ya[te], p))
        praucs.append(average_precision_score(ya[te], p))
        n_feats.append(model[:-1].transform(Xa[te][:5]).shape[1])   # features the classifier saw

    return {"antibiotic": antibiotic, "auc": np.mean(aucs), "auc_std": np.std(aucs),
            "prauc": np.mean(praucs), "n_features": int(np.mean(n_feats))}

## Member 2: Grouped Feature Optimization Method

Implements the localized grouped feature selection approach using ANOVA F-statistics. This reducer divides the 6,000 spectral bins into windows of 50, ranks them by predictive power on the training folds, selects the top 30 regions (1,500 bins), and evaluates performance across Member 2's 4 assigned antibiotics: **Ciprofloxacin, Piperacillin-Tazobactam, Norfloxacin, and Ceftazidime**. Results are automatically saved to `results_binary_pso_grouped.csv`.

In [3]:
class GroupedOptimizationSelect(BaseEstimator, TransformerMixin):
    def __init__(self, n_bins_per_group=50, top_k_groups=30):
        self.n_bins_per_group = n_bins_per_group  
        self.top_k_groups = top_k_groups          

    def fit(self, X, y):
        n_samples, n_features = X.shape
        n_groups = n_features // self.n_bins_per_group
        
        group_scores = []
        for i in range(n_groups):
            start_idx = i * self.n_bins_per_group
            end_idx = (i + 1) * self.n_bins_per_group
            X_group = X[:, start_idx:end_idx]
            
            scores = f_classif(X_group, y)[0]
            score = np.nanmean(np.nan_to_num(scores))
            group_scores.append((score, start_idx, end_idx))
            
        group_scores.sort(key=lambda x: x[0], reverse=True)
        self.selected_intervals_ = group_scores[:self.top_k_groups]
        
        self.mask_ = np.zeros(n_features, dtype=bool)
        for score, start, end in self.selected_intervals_:
            self.mask_[start:end] = True
            
        return self

    def transform(self, X):
        return X[:, self.mask_]

# Initialize your reducer method
METHOD_NAME = "binary_pso_grouped"
my_reducer = GroupedOptimizationSelect(n_bins_per_group=50, top_k_groups=30)

rows = []
for abx in MODEL_ABX:
    if abx not in lab.columns:
        print(f"Skipping {abx}: not found in labels.")
        continue
    
    r = run_cv(my_reducer, abx)
    r["method"] = METHOD_NAME
    rows.append(r)
    print(f"Done -> {abx} | AUC = {r['auc']:.3f} | PR-AUC = {r['prauc']:.3f} | Features = {r['n_features']}")

# Save output CSV file
res_member2 = pd.DataFrame(rows)[["antibiotic", "method", "auc", "auc_std", "prauc", "n_features"]]
output_filename = f"results_{METHOD_NAME}.csv"
res_member2.to_csv(output_filename, index=False)

print(f"\nSuccessfully saved results to {output_filename} in your workspace folder!")
display(res_member2)

Done -> Ciprofloxacin | AUC = 0.771 | PR-AUC = 0.593 | Features = 1500
Done -> Piperacillin-Tazobactam | AUC = 0.684 | PR-AUC = 0.161 | Features = 1500
Done -> Norfloxacin | AUC = 0.766 | PR-AUC = 0.571 | Features = 1500
Done -> Ceftazidime | AUC = 0.776 | PR-AUC = 0.377 | Features = 1500

Successfully saved results to results_binary_pso_grouped.csv in your workspace folder!


,antibiotic,method,auc,auc_std,prauc,n_features
0,Ciprofloxacin,binary_pso_grouped,0.770619,0.014895,0.593419,1500
1,Piperacillin-Tazobactam,binary_pso_grouped,0.683537,0.021414,0.160573,1500
2,Norfloxacin,binary_pso_grouped,0.765833,0.027961,0.571190,1500
3,Ceftazidime,binary_pso_grouped,0.775662,0.014932,0.377499,1500
